In [9]:
from MilvusHelper import MilvusHelper
mh = MilvusHelper(host = "localhost", port = "19530", collection_name = "expert_collection")

In [10]:
from datasets import load_from_disk

dataset = load_from_disk("D:/PycharmProjects/TrustDataFilter/tool/DataProcess/embedding_CNN_1")


In [4]:
import pandas as pd
df = pd.DataFrame(dataset)
df.to_csv("embedding_CNN_1.csv", index=False)

In [11]:
# Dataset({
#    features: ['summary', 'document', 'id', 'embedding', 'dim'],
#    num_rows: 5000
#})
mh.add_data(summary=dataset['summary'][0:10], document=dataset['document'][0:10],dims=['dim'][0:10],embedding=dataset['embedding'][0:10])

RPC error: [batch_insert], <DataNotMatchException: (code=1, message=The Input data type is inconsistent with defined schema, please check it.)>, <Time:{'RPC start': '2024-08-07 15:12:00.562240', 'RPC error': '2024-08-07 15:12:00.581962'}>


DataNotMatchException: <DataNotMatchException: (code=1, message=The Input data type is inconsistent with defined schema, please check it.)>

RPC error: [create_collection], <MilvusException: (code=65535, message=type param(max_length) should be specified for varChar field of collection expert_dataset)>, <Time:{'RPC start': '2024-07-15 23:57:42.528368', 'RPC error': '2024-07-15 23:57:42.529950'}>


MilvusException: <MilvusException: (code=65535, message=type param(max_length) should be specified for varChar field of collection expert_dataset)>

RPC error: [query], <MilvusException: (code=65535, message=empty expression should be used with limit)>, <Time:{'RPC start': '2024-10-11 11:12:34.723439', 'RPC error': '2024-10-11 11:12:34.725391'}>


MilvusException: <MilvusException: (code=65535, message=empty expression should be used with limit)>

RPC error: [query], <MilvusException: (code=65535, message=invalid max query result window, (offset+limit) should be in range [1, 16384], but got 32768)>, <Time:{'RPC start': '2024-10-11 11:38:52.869791', 'RPC error': '2024-10-11 11:38:52.870320'}>


MilvusException: <MilvusException: (code=65535, message=invalid max query result window, (offset+limit) should be in range [1, 16384], but got 32768)>

In [40]:
from pymilvus import connections, Collection
import numpy as np

# 连接到 Milvus
connections.connect(alias="default", host="localhost", port="19530")

# 获取指定的 collection
collection_name = "biological_iterate_exp5_gpt35"
collection = Collection(collection_name)

# 搜索相似的嵌入
def find_duplicate_knowledge(threshold=0.95, max_window_size=5000):
    # 加载 collection
    collection.load()

    # 设置搜索参数
    search_params = {
        "metric_type": "COSINE",  # 使用余弦相似度
        "params": {"nprobe": 10}
    }

    offset = 0
    duplicate_knowledge_pairs = []
    total_records = collection.num_entities  # 获取总记录数

    # 分批次查询所有数据
    results = collection.query(
        expr="flag == 1",
        output_fields=["knowledge", "embedding", "flag", "id"],
    )

    # 提取知识点和嵌入向量
    embeddings = np.array([res["embedding"] for res in results])
    knowledge_list = [res["knowledge"] for res in results]
    flag_list = [res["flag"] for res in results]
    id_list = [res["id"] for res in results]

    # 将查询结果转换为字典，方便通过 ID 查找
id_to_knowledge = {res["id"]: (res["knowledge"], res["flag"]) for res in results}

    # 遍历嵌入向量，查找相似度大于阈值的向量
    for i, embedding in enumerate(embeddings):
        query_embedding = [embedding]

        # 搜索相似的嵌入
        search_results = collection.search(
            data=query_embedding,
            anns_field="embedding",
            param=search_params,
            output_fields=["id", "knowledge", "flag"],
            limit=1,  # 可根据需要调整限制返回的结果数
        )

        # 遍历搜索结果，找出重复项，排除自身 (根据id判断)
        for result in search_results[0]:
            if result.entity.get('id') != id_list[i] and knowledge_list[i] == result.entity.get('knowledge'):
                # result.entity.get('id') != id_list[i] and
                # 从 id_to_knowledge 中查找对应的 knowledge 和 flag
                duplicate_knowledge_pairs.append(
                    (knowledge_list[i], result.entity.get('knowledge'),id_list[i], result.entity.get('id'))
                )
    print(id_list[-1])
    return duplicate_knowledge_pairs

# 设置相似度阈值，查找重复的 knowledge
threshold = 0.99  # 可根据实际情况调整
duplicates = find_duplicate_knowledge(threshold)

# 输出重复的 knowledge 对及其 flag 值
# for knowledge1, knowledge2, flag1, flag2 in duplicates:
#     print(f"Duplicate knowledge found: '{knowledge1}' (flag: {flag1}) and '{knowledge2}' (flag: {flag2})")

len(duplicates)

453107952126592723


351

In [19]:
print(duplicates)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



UnicodeEncodeError: 'gbk' codec can't encode character '\xef' in position 113: illegal multibyte sequence

In [34]:
with open("duplicate_knowledge_output.txt", "w", encoding="utf-8") as file:
    for knowledge1, knowledge2, flag1, flag2 in duplicates:
        file.write(f"Duplicate knowledge found: '{knowledge1}' (flag: {flag1}) and '{knowledge2}' (flag: {flag2})\n")
print("Results written to duplicate_knowledge_output.txt")


Results written to duplicate_knowledge_output.txt


In [22]:
len(duplicates)

21720

In [ ]:
from pymilvus import connections, Collection
import json

# 假设 decisionTree 是你定义的函数
def decisionTree(result_reasonable, result_conflict):
    # 模拟你的决策树逻辑，返回 1 表示满足条件
    return result_reasonable > result_conflict

# 连接到 Milvus
connections.connect(alias="default", host="localhost", port="19530")

# 获取指定的 collection
collection_name = "your_collection_name"
collection = Collection(collection_name)

# 读取所有记录
def read_milvus_records():
    collection.load()

    # 查询所有字段
    results = collection.query(expr="", output_fields=[
        "id", "embedding", "knowledge", "topic", "flag",
        "confidence_score", "contradiction_score", "relation_score"
    ])

    return results

# 处理并保存满足条件的记录
def process_and_save_records(records, filename="filtered_records.json"):
    filtered_records = []

    # 遍历所有记录
    for record in records:
        confidence_score = record["confidence_score"]
        contradiction_score = record["contradiction_score"]

        # 调用决策树函数
        if decisionTree(confidence_score, contradiction_score) == 1:
            filtered_records.append(record)

    # 将结果保存到文件
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(filtered_records, f, ensure_ascii=False, indent=4)

    print(f"Filtered records saved to {filename}")

# 主程序
if __name__ == "__main__":
    # 读取所有 Milvus 记录
    all_records = read_milvus_records()

    # 处理并保存满足条件的记录
    process_and_save_records(all_records)


In [12]:
from pymilvus import connections, Collection
import json
import pandas as pd
import pickle
with open(r'D:\PycharmProjects\TDFilter\DecisionTree\model.pkl', 'rb') as file:
        clf_loaded = pickle.load(file)
def decisionTree(reasonable_score,conflict_score):
    # 1. 使用 pickle 加载模型

    # 2. 构造要预测的数据
    # 新的数据，reasonable_score=1 和 conflict_score=1，使用 DataFrame 并指定列名
    X_new = pd.DataFrame({'reasonable_score': [reasonable_score], 'conflict_score': [conflict_score]})

    # 3. 使用加载的模型进行预测
    y_pred = clf_loaded.predict(X_new)

    return y_pred[0]
# 假设 decisionTree 是你定义的函数
# def decisionTree(result_reasonable, result_conflict):
#     # 模拟你的决策树逻辑，返回 1 表示满足条件
#     return result_reasonable > result_conflict

# 连接到 Milvus
connections.connect(alias="default", host="localhost", port="19530")

# 获取指定的 collection
collection_name = "biological_iterate_exp5_gpt35"
collection = Collection(collection_name)

# 读取所有记录
def read_milvus_records():
    collection.load()

    # 查询所有字段
    results = collection.query(expr="flag == 1", output_fields=[
        "flag",
        "confidence_score", "contradiction_score"
    ])

    return results

# 处理并保存满足条件的记录
def process_and_save_records(records, filename="filtered_records.json"):
    filtered_records = []

    # 遍历所有记录
    for record in records:
        record["confidence_score"]= float(record["confidence_score"])
        record["contradiction_score"]= float(record["contradiction_score"])
        confidence_score = record["confidence_score"]
        contradiction_score = record["contradiction_score"]

        # 调用决策树函数
        if decisionTree(confidence_score, contradiction_score) == 1:
            filtered_records.append(record)

    # 将结果保存到文件
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(filtered_records, f, ensure_ascii=False, indent=4)

    print(f"Filtered records saved to {filename}")

# 主程序
if __name__ == "__main__":
    # 读取所有 Milvus 记录
    all_records = read_milvus_records()

    # 处理并保存满足条件的记录
    process_and_save_records(all_records)


Filtered records saved to filtered_records.json


In [13]:
with open("filtered_records.json", "r", encoding="utf-8") as file:
    js = json.load(file)

len(js)

17592

In [ ]:
from pymilvus import connections, Collection
import numpy as np

# 连接到 Milvus
connections.connect(alias="default", host="localhost", port="19530")

# 获取指定的 collection
collection_name = "biological_iterate_exp5_gpt35"
collection = Collection(collection_name)

# 搜索相似的嵌入
def find_duplicate_knowledge(threshold=0.99, max_window_size=16384):
    # 加载 collection
    collection.load()

    # 设置搜索参数
    search_params = {
        "metric_type": "COSINE",  # 使用余弦相似度
        "params": {"nprobe": 10}
    }

    offset = 0
    duplicate_knowledge_pairs = []
    total_records = collection.num_entities  # 获取总记录数

    # 分批次查询所有数据
    while offset < total_records:
        # 确保每次查询的 offset 和 limit 的和不超过 16384
        available_limit = max_window_size - offset
        current_limit = min(available_limit, max_window_size, total_records - offset)

        # 查询指定范围的数据
        results = collection.query(
            expr="",
            output_fields=["knowledge", "embedding", "flag", "id"],
            limit=current_limit,
            offset=offset
        )

        if not results:
            break  # 如果没有更多结果，跳出循环

        # 提取知识点和嵌入向量
        embeddings = np.array([res["embedding"] for res in results])
        knowledge_list = [res["knowledge"] for res in results]
        flag_list = [res["flag"] for res in results]
        id_list = [res["id"] for res in results]

        # 将查询结果转换为字典，方便通过 ID 查找
        id_to_knowledge = {res["id"]: (res["knowledge"], res["flag"]) for res in results}

        # 遍历嵌入向量，查找相似度大于阈值的向量
        for i, embedding in enumerate(embeddings):
            query_embedding = [embedding]

            # 搜索相似的嵌入
            search_results = collection.search(
                data=query_embedding,
                anns_field="embedding",
                param=search_params,
                limit=10,  # 可根据需要调整限制返回的结果数
            )

            # 遍历搜索结果，找出重复项，排除自身 (根据id判断)
            for result in search_results[0]:
                if result.id != id_list[i] and result.distance > threshold:
                    # 从 id_to_knowledge 中查找对应的 knowledge 和 flag
                    if result.id in id_to_knowledge:
                        knowledge2, flag2 = id_to_knowledge[result.id]
                        duplicate_knowledge_pairs.append(
                            (knowledge_list[i], knowledge2, flag_list[i], flag2)
                        )

        # 更新 offset，确保 offset + limit 总和不会超出范围
        offset += current_limit

    return duplicate_knowledge_pairs

# 设置相似度阈值，查找重复的 knowledge
threshold = 0.95  # 可根据实际情况调整
duplicates = find_duplicate_knowledge(threshold)

# 输出重复的 knowledge 对及其 flag 值
for knowledge1, knowledge2, flag1, flag2 in duplicates:
    print(f"Duplicate knowledge found: '{knowledge1}' (flag: {flag1}) and '{knowledge2}' (flag: {flag2})")
